In [1]:
from numba import njit, prange
import numpy as np
import faiss

index_ = [0, 0, 0, 0]
# index_ = [325, 80, 47, 149]
data = np.load(
    f"/home/dhem/workspace/2024.3/data/save/train_test-{index_[0]}-{index_[1]}-{index_[2]}-{index_[3]}.npz"
)

x = data["x"]
y = data["y"]
w = data["w"]
coor = data["coor"]
name = data["name"]

In [2]:
(num_sample, dim_sample) = x.shape
res = faiss.StandardGpuResources()
flat_config = faiss.GpuIndexFlatConfig()
flat_config.device = 0
index = faiss.GpuIndexFlatL2(res, dim_sample, flat_config)
index.add(x)

In [3]:
b = x.copy()
min_number = 40
distances, indices = index.search(b, min_number)

var_y = np.var(np.einsum("ij,i->ij", y[indices], x[:, 0] * w), axis=1)

sum_of_distances = np.sum(distances, axis=1)
sum_of_distances_reverse = np.mean(sum_of_distances) / (
    sum_of_distances + 1e-4 * min_number
)

argsort_ = np.argsort(sum_of_distances_reverse * var_y)[::-1][:100]

energy = np.einsum("ij,i->ij", y[indices[argsort_]], (x[:, 0] * w)[argsort_])
distances = distances[argsort_]
name = name[argsort_]

In [4]:
# (sum_of_distances_reverse * var_y)[argsort_]
index_check = 30
print(
    np.sum(
        (x[indices[argsort_[index_check]]] - x[indices[argsort_[index_check]]][0]) ** 2,
        axis=1,
    )
)
print(distances[index_check])
print(np.max(np.abs((energy[index_check] - energy[index_check][0]) * 627.509)))
print()

[0.00000000e+00 8.54148827e-06 8.93542727e-06 2.53950213e-05
 3.38157314e-05 3.53287971e-05 3.69836915e-05 4.20262607e-05
 4.24756613e-05 4.43440055e-05 4.72242494e-05 5.22613241e-05
 5.98437015e-05 7.46243023e-05 7.63002018e-05 8.33702176e-05
 8.79453345e-05 9.17944472e-05 9.26077369e-05 1.01610890e-04
 1.01704239e-04 1.02749170e-04 1.04371035e-04 1.05968376e-04
 1.13035504e-04 1.13076470e-04 1.14653561e-04 1.15456397e-04
 1.22625233e-04 1.28838223e-04 1.29250873e-04 1.29708733e-04
 1.29842813e-04 1.31211181e-04 1.31405923e-04 1.35407312e-04
 1.37397073e-04 1.37911814e-04 1.40744822e-04 1.41147728e-04]
[0.00000000e+00 8.53743404e-06 8.93883407e-06 2.53999606e-05
 3.38191167e-05 3.53315845e-05 3.69828194e-05 4.20259312e-05
 4.24776226e-05 4.43458557e-05 4.72227111e-05 5.22593036e-05
 5.98458573e-05 7.46278092e-05 7.63004646e-05 8.33692029e-05
 8.79410654e-05 9.17920843e-05 9.26041976e-05 1.01608224e-04
 1.01701356e-04 1.02747232e-04 1.04375184e-04 1.05970539e-04
 1.13029964e-04 1.13070

In [5]:
from matplotlib import pyplot as plt

color_dict = {
    "methane": "#004D40",
    "ethane": "#1A237E",
    "ethylene": "#212121",
    "acetylene": "#7B1FA2",
}

plt.rcParams["figure.figsize"] = np.array([3, 3]) * 520 / 72

f, axes = plt.subplots(10, 10)
axes = axes.reshape(10, 10)

begin_y = 0.025
end_y = 0.95
int_y = 0.0
begin_x = 0.025
end_x = 0.95
int_x = 0.0
end_x += int_x
end_y += int_y

shapexy = np.shape(axes)
inter_x = np.linspace(begin_x, end_x, shapexy[1] + 1)
inter_y = np.linspace(begin_y, end_y, shapexy[0] + 1)

delta_x = inter_x[1] - inter_x[0] - int_x
delta_y = inter_y[1] - inter_y[0] - int_y

for i in range(shapexy[0]):
    for j in range(shapexy[1]):
        axes[i][j].set_position(
            [
                inter_x[j],
                inter_y[i],
                inter_x[j + 1] - inter_x[j] - int_x,
                inter_y[i + 1] - inter_y[i] - int_y,
            ]
        )
        axes[i][j].xaxis.set_tick_params(
            direction="in", which="both", bottom=True, top=True
        )
        axes[i][j].yaxis.set_tick_params(
            direction="in", which="both", left=True, right=True
        )
        
        axes[i, j].set_xlim(-0.0001, 0.0011)
        axes[i, j].set_ylim(-0.003, 0.033)
        # axes[i, j].set_ylim(-0.002, 0.022)
        
        if i != 0:
            axes[i, j].set_xticks([])
        else:
            axes[i, j].set_xticks([0, 0.001])
            axes[i, j].set_xticklabels([0, 0.001])
        if j != 0:
            axes[i][j].set_yticks([])


for i in range(argsort_.shape[0]):
    axes_i, axes_j = np.unravel_index(i, (10, 10))
    for j in range(indices.shape[1]):
        axes[axes_i, axes_j].scatter(
            distances[i][j],
            np.abs(energy[i][j] - energy[i][0]) * 627.509,
            c=(
                color_dict[name[indices[argsort_[i]]][j].split("_")[0]],
            ),
        )

    axes[axes_i, axes_j].text(
        0.01,
        1-0.01,
        f"{i}",
        transform=axes[axes_i, axes_j].transAxes,
        va="top",
    )
plt.savefig("test.pdf", dpi=300)
plt.clf()

<Figure size 2166.67x2166.67 with 0 Axes>